# Prepare Tabula Sapiens v2 aggregation for HakaseAI L3 Stage 3

**Purpose** — preprocess Tabula Sapiens v2 (Tabula Sapiens Consortium, *Science* 2022 + 2024 v2 update) into a small, query-friendly Parquet that the Replit-hosted `ai-service` loads at runtime to power the **L3 Stage 3 cell-type aggregation** endpoint (`/predict/celltype-aggregation`).

**Why off-platform?** The raw `h5ad` is ~3 GB and CELLxGENE Census needs ~16–32 GB RAM to slice. Replit dev containers can't comfortably do that. We do the heavy lift here in Colab (recommend a *high-RAM* runtime), emit a ~50–150 MB Parquet, then upload that Parquet to `artifacts/ai-service/cache/tabula_sapiens_aggregated.parquet` in the Replit project.

**Output schema (every column required — the loader rejects Parquets missing any of these):**

| column | type | meaning |
| --- | --- | --- |
| `gene_symbol` | str (uppercase) | HUGO symbol |
| `cell_type` | str | `cell_ontology_class` label |
| `tissue` | str | `tissue_in_publication` |
| `organ` | str | organ grouping (this notebook computes it) |
| `mean_expression` | float32 | mean of `log1p(normalized_count)` across cells in the group |
| `n_cells` | int32 | cells contributing to the group |
| `pct_expressing` | float32 | fraction of cells with expression > 0 (0..1) |

**Provenance** — we attach `atlas_name`, `atlas_sha`, `preparation_date`, `n_genes`, `n_cell_types`, `n_organs`, `notebook_version` to the Parquet's pyarrow metadata so the running service can echo them back honestly.

## 1 — Install dependencies

On a Colab Pro **High-RAM** runtime this takes ~3–5 min.

In [ ]:
!pip install --quiet 'cellxgene-census>=1.15' 'scanpy>=1.10' 'anndata>=0.10' 'pyarrow>=16' 'pandas>=2.2' 'numpy<2'

In [ ]:
import os, gc, json, datetime as _dt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import cellxgene_census
import pyarrow as pa
import pyarrow.parquet as pq

NOTEBOOK_VERSION = 'prepare_tabula_sapiens.ipynb v1.0 (Phase 2)'
PREP_DATE = _dt.datetime.now(_dt.timezone.utc).isoformat()
print('Notebook:', NOTEBOOK_VERSION)
print('Date:', PREP_DATE)

## 2 — Pin a CELLxGENE Census release (reproducibility)

We pin to a stable Census release so the SHA is reproducible. List versions with `cellxgene_census.get_census_version_directory()` and pick the latest stable, then hard-pin its SHA below.

In [ ]:
# Inspect available pinned releases
directory = cellxgene_census.get_census_version_directory()
stable_versions = {k: v for k, v in directory.items() if v.get('release_build') == 'stable' or 'stable' in v.get('alias', '')}
for name, meta in list(stable_versions.items())[:10]:
    print(name, '->', meta.get('release_date'), '|', meta.get('alias'))

In [ ]:
# Pick a release. Update CENSUS_VERSION to the dated alias you want to pin.
CENSUS_VERSION = 'stable'   # safer: replace with an explicit dated tag like '2024-07-01' for full reproducibility
census = cellxgene_census.open_soma(census_version=CENSUS_VERSION)
print('Opened Census:', CENSUS_VERSION)

## 3 — Pull Tabula Sapiens v2 cells (homo sapiens, dataset filter)

Tabula Sapiens v2 is exposed in CELLxGENE Census via dataset titles like `"Tabula Sapiens - All Cells"` (or per-organ datasets). We slice by `dataset_title` so we get the consortium's curated cell-type and tissue labels.

In [ ]:
# Probe dataset titles. Adjust the filter if Tabula Sapiens v2 surfaces under a different title in your Census release.
datasets = census['census_info']['datasets'].read().concat().to_pandas()
ts = datasets[datasets['dataset_title'].str.contains('Tabula Sapiens', case=False, na=False)]
print(ts[['dataset_id', 'dataset_title', 'dataset_total_cell_count']].to_string(index=False))

In [ ]:
# Pick a dataset_id (or a title-substring filter). For 'all cells' use the consortium-wide one.
DATASET_ID = ts.iloc[0]['dataset_id']   # replace with the chosen dataset_id
ATLAS_NAME = ts.iloc[0]['dataset_title']
ATLAS_SHA = DATASET_ID
print('Using:', ATLAS_NAME, '->', DATASET_ID)

In [ ]:
# Fetch as anndata. obs_value_filter pins the dataset; we only pull the columns we need
# in obs to keep memory reasonable.
adata = cellxgene_census.get_anndata(
    census=census,
    organism='Homo sapiens',
    obs_value_filter=f'dataset_id == "{DATASET_ID}"',
    obs_column_names=['cell_type', 'tissue', 'tissue_general', 'assay'],
)
print('Cells:', adata.n_obs, '| Genes:', adata.n_vars)
print(adata.obs.head())

## 4 — Normalize and log1p

Standard `scanpy` normalization to 10k counts per cell, then `log1p`. This matches the original Tabula Sapiens preprocessing convention.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print('Normalized + log1p done.')

## 5 — Aggregate to (gene_symbol, cell_type, tissue) means

We compute, for each `(cell_type, tissue)` group, **per-gene mean log-expression** and **percent expressing**. This is the canonical Drug2cell-style aggregation.

In [ ]:
# Group key
obs = adata.obs.copy()
obs['group_key'] = obs['cell_type'].astype(str) + ' || ' + obs['tissue'].astype(str)
groups = obs.groupby('group_key', observed=True).indices
print('Groups:', len(groups))

In [ ]:
# Use sparse matrix for speed; compute per-group mean and pct_expressing per gene.
import scipy.sparse as sp
X = adata.X if sp.issparse(adata.X) else sp.csr_matrix(adata.X)
X = X.tocsr()

# Resolve gene symbols. CELLxGENE Census stores Ensembl IDs in var.index by default; the
# 'feature_name' column carries the HUGO symbol when present.
var = adata.var.copy()
if 'feature_name' in var.columns:
    gene_symbols = var['feature_name'].astype(str).str.upper().values
else:
    gene_symbols = var.index.astype(str).str.upper().values

rows = []
for key, idx in groups.items():
    sub = X[idx, :]
    n_cells = sub.shape[0]
    # per-gene mean log1p expression
    mean_expr = np.asarray(sub.mean(axis=0)).ravel()
    # per-gene fraction of cells with non-zero expression
    nonzero = (sub > 0).sum(axis=0)
    pct_expr = np.asarray(nonzero).ravel() / max(n_cells, 1)
    cell_type, tissue = key.split(' || ', 1)
    # Keep only genes with non-trivial expression in this group to keep Parquet small
    keep = (mean_expr > 0.05) | (pct_expr > 0.10)
    if not keep.any():
        continue
    rows.append(pd.DataFrame({
        'gene_symbol': gene_symbols[keep],
        'cell_type': cell_type,
        'tissue': tissue,
        'mean_expression': mean_expr[keep].astype(np.float32),
        'n_cells': np.int32(n_cells),
        'pct_expressing': pct_expr[keep].astype(np.float32),
    }))

agg = pd.concat(rows, ignore_index=True)
print('Aggregated rows:', len(agg))
print(agg.head())
del rows; gc.collect()

## 6 — Map tissue → organ

Roll up tissue labels into ~25–30 organ buckets so the L3 UI can show organ-level summaries. Edit `TISSUE_TO_ORGAN` to fit the dataset you pull — anything unmapped falls back to `tissue_general` from the Census obs.

In [ ]:
TISSUE_TO_ORGAN = {
    'liver': 'Liver', 'gallbladder': 'Liver',
    'kidney': 'Kidney', 'cortex of kidney': 'Kidney', 'medulla of kidney': 'Kidney',
    'heart': 'Heart', 'cardiac atrium': 'Heart', 'cardiac ventricle': 'Heart',
    'lung': 'Lung', 'trachea': 'Lung',
    'small intestine': 'Intestine', 'large intestine': 'Intestine', 'colon': 'Intestine',
    'stomach': 'Stomach',
    'pancreas': 'Pancreas',
    'spleen': 'Spleen',
    'thymus': 'Thymus',
    'lymph node': 'Lymphoid',
    'bone marrow': 'Bone marrow',
    'blood': 'Blood',
    'skin of body': 'Skin', 'skin': 'Skin',
    'muscle of pelvic diaphragm': 'Muscle', 'skeletal muscle tissue': 'Muscle',
    'adipose tissue': 'Adipose',
    'brain': 'Brain', 'cerebral cortex': 'Brain', 'hippocampal formation': 'Brain',
    'spinal cord': 'CNS',
    'eye': 'Eye', 'cornea': 'Eye', 'retina': 'Eye',
    'mammary gland': 'Mammary',
    'uterus': 'Reproductive', 'ovary': 'Reproductive', 'fallopian tube': 'Reproductive',
    'prostate gland': 'Reproductive', 'testis': 'Reproductive',
    'tongue': 'Oral', 'salivary gland': 'Oral',
    'bladder organ': 'Bladder', 'urinary bladder': 'Bladder',
}

def to_organ(t: str) -> str:
    t = (t or '').lower().strip()
    if t in TISSUE_TO_ORGAN:
        return TISSUE_TO_ORGAN[t]
    # fallback heuristic
    for key, val in TISSUE_TO_ORGAN.items():
        if key in t:
            return val
    return 'Other'

agg['organ'] = agg['tissue'].astype(str).map(to_organ)
print(agg['organ'].value_counts().head(20))

## 7 — Write Parquet with provenance metadata

We write with snappy compression and embed the atlas SHA + cell/gene/organ counts + notebook version in pyarrow file metadata. The Replit loader echoes these back via `model_info.atlas`.

In [ ]:
OUT_PATH = '/content/tabula_sapiens_aggregated.parquet'

# Reorder columns to match REQUIRED_COLUMNS in celltype_aggregation.py
agg = agg[['gene_symbol', 'cell_type', 'tissue', 'organ', 'mean_expression', 'n_cells', 'pct_expressing']]

tbl = pa.Table.from_pandas(agg, preserve_index=False)

metadata = {
    'atlas_name': str(ATLAS_NAME),
    'atlas_sha': str(ATLAS_SHA),
    'preparation_date': PREP_DATE,
    'n_genes': str(int(agg['gene_symbol'].nunique())),
    'n_cell_types': str(int(agg['cell_type'].nunique())),
    'n_organs': str(int(agg['organ'].nunique())),
    'notebook_version': NOTEBOOK_VERSION,
    'census_version': str(CENSUS_VERSION),
}
tbl = tbl.replace_schema_metadata({**(tbl.schema.metadata or {}), **{k.encode(): v.encode() for k, v in metadata.items()}})

pq.write_table(tbl, OUT_PATH, compression='snappy')
size_mb = os.path.getsize(OUT_PATH) / (1024 * 1024)
print(f'Wrote {OUT_PATH} ({size_mb:.1f} MB)')
print('Metadata:', json.dumps(metadata, indent=2))

## 8 — Download the Parquet and upload to Replit

Run the cell below to get a download. Then in Replit:

1. Place the file at `artifacts/ai-service/cache/tabula_sapiens_aggregated.parquet`.
2. Restart the **AI Service (Python)** workflow.
3. Verify with: `curl http://localhost:8090/models/status` — `celltype_aggregation.loaded` should flip to `true` and provenance should match what we wrote here.
4. End-to-end test: `curl -s -X POST http://localhost:8080/api/ai/celltype-aggregation -H 'Content-Type: application/json' -d '{"targets":[{"gene_symbol":"EGFR"}]}'`.

In [ ]:
from google.colab import files  # type: ignore
files.download(OUT_PATH)